## Training 3D segmentation model end-to-end

This is not the final model but only sanity checks that a 3D segmentation end-to-end pipeline works correctly

In [11]:
import torch
from monai.data import DataLoader
from monai.apps import DecathlonDataset
from src.data.transforms import get_base_transformations

from monai.transforms import (Compose, ScaleIntensityRanged, EnsureTyped)
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric

In [2]:
# Set up of device
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: mps


In [3]:
transforms = get_base_transformations()

train_ds = DecathlonDataset(
    root_dir="../data",
    task="Task01_BrainTumour",
    section="training",
    transform=transforms,
    download=False
)

/Users/spence/Documents/Projects/Python/BraTS/.venv/lib/python3.12/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)
Loading dataset: 100%|██████████| 388/388 [01:16<00:00,  5.04it/s]


In [7]:
# Additional preprocessing
train_runtime_transforms = Compose([
    ScaleIntensityRanged(
            keys=["image"],
            a_min=0, a_max=3000,
            b_min=0, b_max=1,
            clip=True
        ),
        EnsureTyped(keys=["label"], dtype=torch.long),
])

In [8]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

In [10]:
model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(16, 32, 64),
    strides=(2, 2),
    num_res_units=1,
).to(device)

In [12]:
# Loss function and metrics
loss_fn = DiceCELoss(
    to_onehot_y=True,
    softmax=True
)

dice_metric = DiceMetric(
    include_background=False,
    reduction="mean"
)

### Training

In [16]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

i = 0
model.train()
epoch_loss = 0.0

for i, batch in enumerate(train_loader):
    if i == 3:   # stop after 3 batches
        break

    batch = train_runtime_transforms(batch)
    inputs = batch["image"].to(device)
    labels = batch["label"].to(device)

    optimizer.zero_grad()
    outputs = model(inputs)
    outputs = outputs[..., :labels.shape[-1]]
    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()
    print(f"Batch {i} loss: {loss.item():.4f}")

print("Avg loss:", epoch_loss / (i + 1))

Batch 0 loss: 2.0913
Batch 1 loss: 2.0700
Batch 2 loss: 1.9713
Avg loss: 1.533153384923935
